# XGBoost. Исходная и очищенная история

Сравниваем `raw` и `clean` на одинаковых признаках, параметрах и датах. В `clean` цель и шесть статусов очищаются до построения признаков. Каждый вариант содержит восемь прямых моделей с обучением на всех получасовых стартах.

Параметры и число деревьев встроены из секции `refit_params` файла `notebooks/xgb_tuning/best_params/best_params.json`. Включите GPU в Kaggle.

## Порядок проверки

1. Обучаем базы на train и получаем прогнозы val из 10:30.
2. На первой половине val оцениваем четыре калибровки, `none`, `global`, `grouped`, `nonlinear`. На второй выбираем способ по WAPE + |Relative Bias| и пересчитываем его коэффициенты на всём val.
3. Обучаем базы заново на train + val и применяем зафиксированную калибровку к test.
4. Сравниваем варианты по исходным ответам test до и после поправки.

Для исходных данных train заканчивается перед 7 октября, val занимает 7-24 октября, test занимает 25-31 октября 2025 года. Test уже использовался в проекте.

Этот запуск заканчивается исторической оценкой. Полное обучение на истории и конкурсные файлы вынесены за его пределы.


In [ ]:
from pathlib import Path
import ast
import gc
import hashlib
import json
import math
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb
try:
    import cupy as cp
except ImportError as error:
    raise RuntimeError('Для GPU-матриц нужен CuPy в окружении Kaggle') from error
from sklearn.isotonic import IsotonicRegression
from IPython.display import display
DATA_DIR = None
CACHE_INPUTS = {'raw': None, 'clean': None}
OUTPUT_DIR = Path('/kaggle/working/xgb_clean_vs_raw')
DEVICE = 'cuda'
N_JOBS = 4
SEED = 42
TEST_DAYS = 7
TRAIN_FRACTION = 0.8
TARGET_SCALE = 100000.0
OBJECTIVE = 'reg:absoluteerror'
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
VARIANTS = ['raw', 'clean']
CALIBRATIONS = ['none', 'global', 'grouped', 'nonlinear']
CLEANING = {'window_days': 28, 'min_days': 14, 'z_threshold': 4.0, 'log_scale_floor': 0.25, 'mad_normalization': 1.4826}
GROUP_SHRINKAGE_ROWS = 2000.0
ISOTONIC_BINS = 128
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PIPELINE_VERSION = 'xgb_clean_vs_raw_mad4_all_halfhours_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
BEST_PARAMS = {'1': {'n_estimators': 2530, 'max_depth': 6, 'learning_rate': 0.036760715868196514, 'min_child_weight': 1.0809355847970479, 'subsample': 0.9003563537429966, 'colsample_bytree': 0.6639976846136139, 'reg_alpha': 9.305271303855188e-05, 'reg_lambda': 7.596418498005285}, '2': {'n_estimators': 1419, 'max_depth': 4, 'learning_rate': 0.10190438495585248, 'min_child_weight': 1.8389812995413264, 'subsample': 0.6817515181500764, 'colsample_bytree': 0.6619351860505892, 'reg_alpha': 0.0022965441587741537, 'reg_lambda': 14.08596091970978}, '3': {'n_estimators': 1930, 'max_depth': 3, 'learning_rate': 0.07005359250881395, 'min_child_weight': 2.2064897603579636, 'subsample': 0.8779096005107611, 'colsample_bytree': 0.7846752865583575, 'reg_alpha': 0.018954349619526665, 'reg_lambda': 1.043977626067855}, '4': {'n_estimators': 1827, 'max_depth': 4, 'learning_rate': 0.02925325002470326, 'min_child_weight': 1.5841239345284504, 'subsample': 0.8126045473410459, 'colsample_bytree': 0.7066552862291448, 'reg_alpha': 2.3192129195348187e-05, 'reg_lambda': 0.22202924731915144}, '5': {'n_estimators': 1387, 'max_depth': 3, 'learning_rate': 0.03164978915749773, 'min_child_weight': 1.9902815543275585, 'subsample': 0.6844098448217815, 'colsample_bytree': 0.7432566543439275, 'reg_alpha': 0.010252186188681373, 'reg_lambda': 0.5427730704771981}, '6': {'n_estimators': 1566, 'max_depth': 3, 'learning_rate': 0.023401679328514332, 'min_child_weight': 3.1806793320017555, 'subsample': 0.87035767368126, 'colsample_bytree': 0.7980364720685608, 'reg_alpha': 0.02123901077609503, 'reg_lambda': 0.3328783942599951}, '7': {'n_estimators': 1136, 'max_depth': 5, 'learning_rate': 0.0330759240922086, 'min_child_weight': 3.8692684642092345, 'subsample': 0.6794525496919193, 'colsample_bytree': 0.6799389555259473, 'reg_alpha': 2.7314479492664567, 'reg_lambda': 0.3796668381307971}, '8': {'n_estimators': 534, 'max_depth': 4, 'learning_rate': 0.09417045025581995, 'min_child_weight': 13.418857993305474, 'subsample': 0.689895840619612, 'colsample_bytree': 0.6918995986906583, 'reg_alpha': 0.3746751674810787, 'reg_lambda': 1.8540896283895405}}
PARAMETER_SOURCE_SHA256 = '855a594510b647bca29fa318d610f05d91823b3946fcf04e1326ecd793a579bc'
CLEANING_IMPLEMENTATION_SHA256 = '61c115a14e0e53d0f948ff5239b7147331261315748f3f997e992b989059876d'


In [ ]:
import ctypes
import warnings
if DEVICE != 'cuda':
    raise ValueError('Этот ноутбук допускает обучение только на GPU: DEVICE должен быть cuda')
if not xgb.build_info().get('USE_CUDA', False):
    raise RuntimeError('Установленная сборка XGBoost не поддерживает CUDA. Нужна CUDA-сборка XGBoost в Kaggle')
try:
    cuda_driver = ctypes.CDLL('libcuda.so.1')
    cuda_status = cuda_driver.cuInit(0)
    gpu_count = ctypes.c_int()
    if cuda_status != 0 or cuda_driver.cuDeviceGetCount(ctypes.byref(gpu_count)) != 0 or gpu_count.value < 1:
        raise RuntimeError('CUDA не видит GPU. Включите GPU accelerator в настройках Kaggle Notebook')
except OSError as error:
    raise RuntimeError('Драйвер NVIDIA недоступен. Включите GPU accelerator в Kaggle') from error
warnings.filterwarnings('error', message='(?s).*(No visible GPU|not compiled with CUDA).*', category=UserWarning)
print(f'CUDA готова. GPU: {gpu_count.value}. XGBoost: {xgb.__version__}')
cp.cuda.Device(0).use()
cp.cuda.get_current_stream().synchronize()
print('Модель и матрицы: cuda:0; входы float32 CuPy')


In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)
if DATA_DIR is None:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('train_solo_track.parquet')]
    if not candidates and Path('data/train_solo_track.parquet').is_file():
        candidates = [Path('data')]
    if len(candidates) != 1:
        raise ValueError(f'Укажите DATA_DIR: найдено {len(candidates)} train parquet')
    DATA_DIR = candidates[0]
TRAIN_PATH = Path(DATA_DIR) / 'train_solo_track.parquet'
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if history.isna().any().any() or history.duplicated(['timestamp', 'route_id']).any():
    raise ValueError('Пропуски или дубликаты')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)) or len(history) != len(times) * len(routes):
    raise ValueError('Нарушена полная получасовая панель')
raw_values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)).copy() for name in [TARGET, *STATUSES]}
if any((not np.isfinite(array).all() or (array < 0).any() for array in raw_values.values())):
    raise ValueError('Нужны конечные неотрицательные исходные значения')
del history
end = times[-1]
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) < TEST_DAYS + 20:
    raise ValueError('Недостаточно полных дат')
before_test = dates[:-TEST_DAYS]
split_index = int(math.floor(len(before_test) * TRAIN_FRACTION))
val_start = before_test[split_index].normalize()
test_start = dates[-TEST_DAYS].normalize()
val_dates = dates[(dates >= val_start) & (dates < test_start)]
test_dates = dates[dates >= test_start]
calibration_select_start = val_dates[len(val_dates) // 2].normalize()
route_levels = np.median(raw_values[TARGET][times < val_start], axis=0)
route_groups = pd.DataFrame({'route_id': routes, 'volume_group': pd.qcut(pd.Series(route_levels).rank(method='first'), min(5, len(routes)), labels=False).to_numpy()})
route_groups.to_csv(RUN_DIR / 'route_volume_groups_train.csv', index=False)
pd.DataFrame({'route_id': routes, 'gpu_category_code': np.arange(len(routes))}).to_csv(RUN_DIR / 'route_encoding.csv', index=False)
route_dtype = pd.CategoricalDtype(categories=routes)
display(pd.DataFrame([{'split': 'train', 'from': str(times[0]), 'before': str(val_start)}, {'split': 'calibration_fit', 'from': str(val_start), 'before': str(calibration_select_start)}, {'split': 'calibration_select', 'from': str(calibration_select_start), 'before': str(test_start)}, {'split': 'train_plus_val_refit', 'from': str(times[0]), 'before': str(test_start)}, {'split': 'test', 'from': str(test_dates[0]), 'through': str(test_dates[-1])}]))


## Причинная очистка только в варианте clean

Тот же фильтр, что в завершённом опыте MAD-4: log1p, предыдущие 28 дней того же получасового слота, минимум 14 наблюдений, медиана и точный MAD одного окна. Масштаб `max(1.4826 × MAD, 0.25)`, порог 4. Выброс заменяется `expm1` медианы. Проверяются верхние/нижние выбросы, включая нули; строки не удаляются. Raw-ветка не проходит фильтр.

Фильтр использует только предыдущие исходные наблюдения. На val/test очищается только уже известная к cutoff входная история; исходные ответы test остаются неизменными. Факты для метрик и калибровки всегда исходные. В начале окна без достаточной истории очистки нет.


In [ ]:
def clean_raw_array(array, timestamps, settings):
    array = np.asarray(array, dtype=np.float32)
    if not np.isfinite(array).all() or (array < 0).any():
        raise ValueError('Для log1p нужны конечные неотрицательные исходные значения')
    cleaned = array.copy()
    flagged = np.zeros(array.shape, dtype=bool)
    slots = timestamps.hour * 2 + timestamps.minute // 30
    for slot in range(48):
        positions = np.flatnonzero(slots == slot)
        logged = np.log1p(array[positions].astype(np.float64))
        window_days = settings['window_days']
        min_days = settings['min_days']
        if not 1 <= min_days <= window_days:
            raise ValueError('Нужно 1 <= min_days <= window_days')
        if len(positions) <= min_days:
            continue
        padded = np.pad(logged, ((window_days, 0), (0, 0)), constant_values=np.nan)
        windows = np.lib.stride_tricks.sliding_window_view(padded, window_days, axis=0)[:-1]
        eligible = np.arange(len(positions)) >= min_days
        past = windows[eligible]
        median = np.nanmedian(past, axis=-1)
        mad = np.nanmedian(np.abs(past - median[..., None]), axis=-1)
        scale = np.maximum(settings['mad_normalization'] * mad, settings['log_scale_floor'])
        mask = np.abs(logged[eligible] - median) > settings['z_threshold'] * scale
        destination = positions[eligible]
        cleaned[destination] = np.where(mask, np.expm1(median), array[destination]).astype(np.float32)
        flagged[destination] = mask
    return (cleaned, flagged)


In [ ]:
FEATURE_CODE = "\n\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\ntarget = values[TARGET]\n\ndef past(array, positions):\n    positions = np.asarray(positions, dtype=np.int64)\n    valid = (positions >= 0) & (positions < len(array))\n    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)\n    result[~valid] = np.nan\n    return result\n\ndef average_available(matrices):\n    total = np.zeros_like(matrices[0], dtype=np.float32)\n    count = np.zeros_like(total)\n    for matrix in matrices:\n        valid = np.isfinite(matrix)\n        total += np.where(valid, matrix, 0)\n        count += valid\n    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)\n\ndef rolling(array, steps):\n    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)\nmeans = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}\nlevel24 = rolling(target, 48)\npositions = np.arange(len(times))\nextra_signals = {}\nfor name in STATUSES[:3]:\n    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)\nextra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))\nextra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))\nfor name in STATUSES[3:]:\n    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)\nextra_signals['hub_buffer'] = values['status_5'] - values['status_4']\nextra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']\nextra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)\nhub_signal = extra_signals['status_6_diff_hourly']\n\ndef common_features(origins):\n    common = {}\n    for k in (0, 1, 2, 4):\n        common[f'target_back_{k}'] = past(target, origins - k)\n    for name, array in values.items():\n        for steps in (4, 12):\n            common[f'{name}_mean_{steps}'] = means[name, steps][origins]\n        if name in STATUSES:\n            common[f'{name}_latest'] = array[origins]\n            if name in STATUSES[:3]:\n                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)\n    for name, array in extra_signals.items():\n        common[name] = array[origins]\n    for lag in (48, 96):\n        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)\n    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)\n    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)\n    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]\n    recent = means[TARGET, 4][origins]\n    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])\n    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)\n    return common\nCORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']\nFEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}\n\ndef calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):\n    grid = pd.DatetimeIndex(timestamps)\n    route_ids = np.asarray(route_ids)\n    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))\n    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):\n        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')\n    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):\n        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')\n    if len(np.unique(route_ids)) != len(route_ids):\n        raise ValueError('Идентификаторы маршрутов повторяются')\n    steps = float(max_lookback_hours) * 2\n    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):\n        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')\n    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:\n        raise ValueError('min_pairs должен быть целым числом >=3')\n    if not 0 <= min_correlation <= 1 or route_block_size < 1:\n        raise ValueError('Некорректные параметры выбора корреляции')\n    max_lag = int(steps)\n    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))\n    anchors = np.arange(max_lag + 2, stop)\n    best_corr = np.full(len(route_ids), np.nan)\n    best_lag = np.full(len(route_ids), np.nan)\n    best_pairs = np.zeros(len(route_ids), dtype=np.int32)\n    for start in range(0, len(route_ids), route_block_size):\n        finish = min(start + route_block_size, len(route_ids))\n        x = target_values[anchors, start:finish].astype(np.float64)\n        for lag in range(2, max_lag + 1):\n            z = status6_diff[anchors - lag, start:finish].astype(np.float64)\n            valid = np.isfinite(x) & np.isfinite(z)\n            count = valid.sum(axis=0)\n            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))\n            divisor = np.maximum(count, 1)\n            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))\n            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor\n            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)\n            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)\n            denominator = np.sqrt(vx * vz)\n            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))\n            corr = np.clip(corr, -1, 1)\n            previous = best_corr[start:finish]\n            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))\n            best_corr[start:finish][better] = corr[better]\n            best_lag[start:finish][better] = lag\n            best_pairs[start:finish][better] = count[better]\n    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)\n    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})\n\ndef add_correlation_features(frame, h, lag_table, chunk_rows=200000):\n    if h not in range(1, 9) or chunk_rows < 1:\n        raise ValueError('Некорректный горизонт или размер блока')\n    by_route = lag_table.set_index('route_id').reindex(routes)\n    route_index = pd.Index(routes)\n    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)\n    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}\n    for name in CORRELATION_COLUMNS:\n        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)\n    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}\n    for start in range(0, len(frame), chunk_rows):\n        finish = min(start + chunk_rows, len(frame))\n        block = frame.iloc[start:finish]\n        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))\n        route = route_index.get_indexer(block.route_id)\n        if (origin < 0).any() or (route < 0).any():\n            raise ValueError('Строка признаков не сопоставлена с историей')\n        selected = lag[route]\n        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)\n        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))\n        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)\n        available &= np.isfinite(signal)\n        for name, array in quality.items():\n            frame.iloc[start:finish, column_positions[name]] = array[route]\n        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)\n        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)\n    return frame\n\ndef horizon_dataset(h, origins):\n    slot = origins + h\n    dates = times[origins] + h * STEP\n    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})\n    for name, matrix in common_features(origins).items():\n        frame[name] = matrix.ravel()\n    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}\n    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)\n    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)\n    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)\n    for name, array in calendar.items():\n        frame[name] = np.repeat(array, len(routes))\n    frame = add_correlation_features(frame, h, validation_lag_table)\n    for days in (1, 2, 7):\n        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()\n    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:\n        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()\n    profile = np.zeros((len(origins), len(routes)))\n    counts = np.zeros_like(profile)\n    for day in range(1, 29):\n        level = past(level24, origins - day * 48)\n        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)\n        profile += np.nan_to_num(adjusted)\n        counts += np.isfinite(adjusted)\n    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()\n    if h == 1:\n        lower = np.zeros_like(target[origins])\n        upper = past(target, origins - 2)\n        for position in (origins - 1, origins):\n            total = past(target, position)\n            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))\n        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()\n        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)\n    else:\n        frame['overlap'] = np.nan\n    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])\n    return frame.astype({name: 'float32' for name in numeric})\n\nbase_horizon_dataset = horizon_dataset\nnetwork_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}\n\ndef ratio_features(current, usual):\n    denominator = np.maximum(usual, np.float32(1.0))\n    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))\n    return ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32)\n\ndef horizon_dataset(h, origins):\n    frame = base_horizon_dataset(h, origins)\n    added = {}\n    slot = origins + h\n    profile = {}\n    for offset in (-2, -1, 0, 1, 2):\n        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])\n        if offset != 0:\n            added[f'slot_daily7_neighbor_{offset:+d}'] = profile[offset].ravel()\n    added['slot_daily7_change_2h'] = (profile[2] - profile[-2]).ravel()\n    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])\n    finite = np.isfinite(stack)\n    masked = np.where(finite, stack, -np.inf)\n    peak = masked.max(axis=0)\n    available = finite.any(axis=0)\n    added['slot_daily7_peak_value'] = np.where(available, peak, np.nan).ravel()\n    added['slot_daily7_peak_offset_minutes'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()\n    for name in STATUSES[:3]:\n        for suffix, array in [('latest', values[name]), ('mean_4', means[name, 4])]:\n            current = array[origins]\n            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n            ratio, small = ratio_features(current, usual)\n            added[f'{name}_{suffix}_vs_usual7'] = (current - usual).ravel()\n            added[f'{name}_{suffix}_ratio_usual7'] = ratio.ravel()\n            added[f'{name}_{suffix}_usual7_small'] = small.ravel()\n    for day in (14, 21, 28):\n        added[f'slot_days_back_{day}'] = past(target, slot - day * 48).ravel()\n    added['slot_week1_minus_week2'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()\n    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])\n    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])\n    added['slot_recent3_minus_previous4'] = (recent - previous).ravel()\n    for name, array in network_values.items():\n        current = array[origins]\n        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])\n        ratio, small = ratio_features(current, usual)\n        features = {'latest': current, 'change_1h': current - past(array, origins - 2),\n            'vs_usual7': current - usual, 'ratio_usual7': ratio, 'usual7_small': small}\n        for suffix, feature in features.items():\n            added[f'network_{name}_{suffix}'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()\n        local = values[name][origins]\n        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)\n        past_total = past(array, origins - 2)\n        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)\n        added[f'network_{name}_route_share'] = share.ravel()\n        added[f'network_{name}_route_share_change_1h'] = (share - old_share).ravel()\n    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)\n"
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame


## Два отдельных кэша признаков

Raw и clean используют разные исходные массивы, лаги и словари корреляционных лагов, но один набор колонок и одинаковые строки. В кэш входят все получасы с размеченной целью до test и только старты 10:30 внутри test. `y` - исходная цель, `y_fit` - цель обучения выбранной ветки. Обе исключены из входов.

Кэши различаются отпечатком исходного parquet, кода признаков, очистки и границ. Совместимые кэши ищутся в Kaggle Input; явно задать их можно через CACHE_INPUTS. Оценочные корреляционные словари используют только train. Для train + val они будут переоценены до test, остальные признаки причинны и не требуют пересборки.


In [ ]:
def cache_complete(folder, expected_fingerprint):
    folder = Path(folder)
    try:
        metadata = json.loads((folder / 'manifest.json').read_text())
        if metadata['fingerprint'] != expected_fingerprint:
            return False
        for h in range(1, 9):
            path = folder / f'history_h{h}.parquet'
            record = json.loads(path.with_suffix('.json').read_text())
            if record['fingerprint'] != expected_fingerprint or record['sha256'] != file_hash(path):
                return False
        return True
    except (OSError, ValueError, KeyError):
        return False
cache_dirs = {}
phase_signals = {}
cache_fingerprints = {}
cleaning_rows = []
input_manifests = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')]
source_hash = file_hash(TRAIN_PATH)
for variant in VARIANTS:
    if variant == 'raw':
        values = raw_values
    else:
        values = {}
        for name in [TARGET, *STATUSES]:
            cleaned, flagged = clean_raw_array(raw_values[name], times, CLEANING)
            values[name] = cleaned
            for split, mask in [('train', times < val_start), ('val', (times >= val_start) & (times < test_start)), ('test_observed_history', times >= test_start)]:
                marks = flagged[mask]
                cleaning_rows.append({'column': name, 'split': split, 'rows': int(marks.size), 'replaced': int(marks.sum()), 'replaced_pct': float(100 * marks.mean()), 'zeros_replaced': int((marks & (raw_values[name][mask] == 0)).sum())})
            del cleaned, flagged
    context = {'np': np, 'pd': pd, 'STEP': STEP, 'TARGET': TARGET, 'STATUSES': STATUSES, 'times': times, 'routes': routes, 'values': values}
    exec(FEATURE_CODE, context)
    lag_table = context['calculate_dynamic_shoulder'](context['target'], context['hub_signal'], times, routes, fit_before=val_start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
    context['validation_lag_table'] = lag_table
    lag_table.to_csv(RUN_DIR / f'{variant}_status6_lags_train.csv', index=False)
    phase_signals[variant] = {'target': context['target'], 'signal': context['hub_signal']}
    spec = {'version': PIPELINE_VERSION, 'variant': variant, 'source_sha256': source_hash, 'feature_code_sha256': hashlib.sha256(FEATURE_CODE.encode()).hexdigest(), 'cleaning': CLEANING if variant == 'clean' else None, 'cleaning_code_sha256': CLEANING_IMPLEMENTATION_SHA256 if variant == 'clean' else None, 'val_start': str(val_start), 'test_start': str(test_start), 'last_test_origin': str(test_dates[-1]), 'lag_settings': [LAG_MAX_LOOKBACK_HOURS, LAG_MIN_PAIRS, LAG_MIN_CORRELATION]}
    fingerprint = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
    cache_fingerprints[variant] = fingerprint
    local_cache = OUTPUT_DIR / 'feature_cache' / fingerprint
    explicit = CACHE_INPUTS.get(variant)
    if explicit is not None:
        if not cache_complete(explicit, fingerprint):
            raise ValueError(f'Несовместимый кэш {variant}')
        folder = Path(explicit)
    else:
        folder = next((p for p in [local_cache, *input_manifests] if cache_complete(p, fingerprint)), local_cache)
    if folder == local_cache:
        folder.mkdir(parents=True, exist_ok=True)
        save_json(folder / 'manifest.json', {'fingerprint': fingerprint, 'spec': spec, 'routes': routes.tolist()})
    cache_dirs[variant] = folder
    print(variant, 'кэш', folder, flush=True)
    for h in range(1, 9):
        path = folder / f'history_h{h}.parquet'
        try:
            record = json.loads(path.with_suffix('.json').read_text())
            ready = record['fingerprint'] == fingerprint and record['sha256'] == file_hash(path)
        except (OSError, ValueError, KeyError):
            ready = False
        if ready:
            print(variant, f'h{h}: готовый кэш', flush=True)
            continue
        origins = np.arange(len(times) - h)
        eligible = (times[origins + h] < test_start) | times[origins].isin(test_dates)
        origins = origins[eligible]
        temporary = path.with_suffix('.partial')
        writer = None
        try:
            for start in range(0, len(origins), 24):
                block = origins[start:start + 24]
                frame = context['horizon_dataset'](h, block)
                frame['y'] = raw_values[TARGET][block + h].ravel()
                frame['y_fit'] = context['target'][block + h].ravel()
                table = pa.Table.from_pandas(frame, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
                writer.write_table(table)
        finally:
            if writer is not None:
                writer.close()
        temporary.replace(path)
        save_json(path.with_suffix('.json'), {'fingerprint': fingerprint, 'rows': len(origins) * len(routes), 'sha256': file_hash(path)})
        print(variant, f'h{h}: {len(origins) * len(routes):,} строк', flush=True)
    del context, values
    gc.collect()
cleaning_report = pd.DataFrame(cleaning_rows)
cleaning_report.to_csv(RUN_DIR / 'cleaning_report.csv', index=False)
display(cleaning_report)
for name in ['raw_values', 'frame', 'table']:
    globals().pop(name, None)
gc.collect()
save_json(RUN_DIR / 'fixed_settings.json', {'params': BEST_PARAMS, 'parameter_source_sha256': PARAMETER_SOURCE_SHA256, 'cleaning': CLEANING, 'cache_fingerprints': cache_fingerprints, 'val_start': str(val_start), 'test_start': str(test_start), 'calibration_select_start': str(calibration_select_start), 'train_plus_val_refit': True, 'full_history_refit': False, 'test_used_for_fit_or_selection': False, 'input_device': 'cuda:0', 'input_dtype': 'float32', 'route_encoding': 'fixed_routes_index_categorical'})


## Этап 1: train → прогнозы val

Все получасовые обучающие метки раньше val. Здесь test не прогнозируется; его строка в промежуточной таблице отмечена как ожидающая train + val. Число деревьев фиксировано из best_params. После каждого fit показываются train и val по исходной цели, включая одинаковый train в 10:30. Модели и построчные OOS-прогнозы val сохраняются.


## Матрицы и модель на GPU

Обучающие матрицы float32 и цели передаются в XGBoost как CuPy-массивы на cuda:0. Обучающая матрица остаётся на GPU до расчёта train-прогноза, затем освобождается. Val/test также прогнозируются из CuPy через inplace_predict; на CPU возвращаются только предсказания для метрик и калибровки. route_id кодируется по общему фиксированному справочнику routes и явно помечается feature_types=c; остальные колонки - q. Порядок колонок сохраняется в feature_columns.json, названия и типы записываются в Booster.

Очистка, подготовка признаков, parquet, метрики и четыре калибровки остаются на CPU. На GPU последовательно загружается один горизонт одной ветки; все горизонты одновременно в VRAM не хранятся. Перед fit выводится объём входных GPU-массивов и свободной памяти. Используется одна GPU cuda:0, даже если доступно две T4. Требуется установленный CuPy.


In [ ]:
def metric_row(frame, prediction, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    error = np.asarray(prediction, dtype=np.float64) - actual
    total = actual.sum()
    if total <= 0 or not np.isfinite(error).all():
        raise ValueError('Метрика неопределена: нулевая сумма цели или неконечный прогноз')
    daily = pd.DataFrame({'day': frame.cutoff.dt.normalize(), 'actual': actual, 'error': error, 'absolute': np.abs(error)}).groupby('day').sum()
    daily = daily.loc[daily.actual > 0]
    wape = np.abs(error).sum() / total
    bias = error.sum() / total
    return {'split': split, 'rows': len(frame), 'MAE': float(np.abs(error).mean()), 'WAPE': float(wape), 'Relative Bias': float(bias), '|Relative Bias|': float(abs(bias)), 'score': float(wape + abs(bias)), 'daily_score': float(((daily.absolute + daily.error.abs()) / daily.actual).mean())}

def require_cuda(model):
    actual = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
    if not actual.startswith('cuda'):
        raise RuntimeError(f'Обучение перешло на {actual}')

def model_columns(frame):
    return frame.columns.drop(['cutoff', 'timestamp', 'y', 'y_fit', 'overlap']).tolist()

def make_model(h, columns):
    types = ['c' if name == 'route_id' else 'q' for name in columns]
    return xgb.XGBRegressor(**BEST_PARAMS[str(h)], objective=OBJECTIVE, eval_metric='mae', tree_method='hist', device=DEVICE, enable_categorical=True, feature_types=types, max_cat_to_onehot=1, random_state=SEED, n_jobs=N_JOBS)

def release_gpu_pool():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

def gpu_matrix(frame, columns):
    host = np.empty((len(frame), len(columns)), dtype=np.float32)
    route_position = columns.index('route_id')
    codes = pd.Index(routes).get_indexer(frame.route_id.astype('int64'))
    if (codes < 0).any():
        raise ValueError('Маршрут отсутствует в фиксированном справочнике')
    host[:, route_position] = codes
    numeric = [name for name in columns if name != 'route_id']
    positions = [index for index, name in enumerate(columns) if name != 'route_id']
    host[:, positions] = frame[numeric].to_numpy(dtype=np.float32)
    matrix = cp.asarray(host, dtype=cp.float32, order='C')
    if matrix.device.id != 0:
        raise RuntimeError('Матрица должна находиться на cuda:0')
    return matrix

def fit_gpu_model(h, data, mask, columns):
    release_gpu_pool()
    matrix = gpu_matrix(data.loc[mask, columns], columns)
    labels = cp.asarray(data.loc[mask, 'y_fit'].to_numpy(dtype=np.float32) / TARGET_SCALE, dtype=cp.float32)
    free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
    print('GPU inputs GiB', round((matrix.nbytes + labels.nbytes) / 2 ** 30, 3), 'free GiB before fit', round(free_bytes / 2 ** 30, 3), flush=True)
    model = make_model(h, columns)
    model.fit(matrix, labels, verbose=False)
    require_cuda(model)
    expected = ['c' if name == 'route_id' else 'q' for name in columns]
    if model.get_booster().feature_types != expected:
        raise RuntimeError('Не сохранена категориальность route_id')
    model.get_booster().feature_names = columns
    return (model, matrix)

def predict_gpu_matrix(model, matrix):
    prediction = model.get_booster().inplace_predict(matrix)
    result = cp.maximum(prediction.astype(cp.float64) * TARGET_SCALE, 0)
    return cp.asnumpy(result)

def predict_gpu_frame(model, frame, columns):
    matrix = gpu_matrix(frame, columns)
    result = predict_gpu_matrix(model, matrix)
    del matrix
    release_gpu_pool()
    return result
feature_columns = {}
calibration_predictions = {}
stage1_rows = []
for variant in VARIANTS:
    prediction_parts = []
    for h in range(1, 9):
        data = pd.read_parquet(cache_dirs[variant] / f'history_h{h}.parquet')
        data['route_id'] = data.route_id.astype(route_dtype)
        columns = model_columns(data)
        if h in feature_columns and feature_columns[h] != columns:
            raise ValueError('Разные входные колонки raw и clean')
        feature_columns[h] = columns
        train = data.timestamp.lt(val_start)
        train1030 = train & data.cutoff.isin(dates[dates < val_start])
        val = data.cutoff.isin(val_dates)
        print('TRAIN', variant, f'h{h}', 'rows', int(train.sum()), 'trees', BEST_PARAMS[str(h)]['n_estimators'], flush=True)
        clock = time.monotonic()
        model, train_gpu = fit_gpu_model(h, data, train, columns)
        model.save_model(RUN_DIR / f'{variant}_h{h}_train_only.ubj')
        train_pred = predict_gpu_matrix(model, train_gpu)
        del train_gpu
        release_gpu_pool()
        rows = [metric_row(data.loc[train], train_pred, 'train_all_halfhours')]
        rows.append(metric_row(data.loc[train1030], train_pred[data.loc[train].cutoff.isin(dates[dates < val_start]).to_numpy()], 'train_1030'))
        part = data.loc[val, ['cutoff', 'timestamp', 'route_id', 'y']].copy()
        part['prediction'] = predict_gpu_frame(model, data.loc[val, columns], columns)
        part['horizon'] = h
        rows.append(metric_row(part, part.prediction, 'val_oos'))
        rows.append({'split': 'test', 'status': 'не оценивается до train_plus_val'})
        table = pd.DataFrame(rows)
        table.insert(0, 'horizon', h)
        table.insert(0, 'variant', variant)
        stage1_rows.append(table)
        display(table)
        prediction_parts.append(part)
        print('DONE TRAIN', variant, f'h{h}', 'seconds', round(time.monotonic() - clock, 1), flush=True)
        del data, model, train_pred
        release_gpu_pool()
    calibration_predictions[variant] = pd.concat(prediction_parts, ignore_index=True)
    calibration_predictions[variant].to_parquet(RUN_DIR / f'{variant}_val_oos.parquet', index=False)
pd.concat(stage1_rows, ignore_index=True).to_csv(RUN_DIR / 'train_only_metrics.csv', index=False)
save_json(RUN_DIR / 'feature_columns.json', feature_columns)


## Выбор и фиксация калибровки по val

Прежние четыре метода: none; общий множитель; множитель по горизонту × train-квинтилю объёма со стягиванием к общему (2 000 псевдострок); изотоническая калибровка на 128 квантильных корзинах с пропорциональной экстраполяцией. Параметры методов не перебираются.

Каждый вариант выбирает тип только по поздней половине val с коэффициентами, обученными на ранней половине. Затем коэффициенты выбранного типа переоцениваются по всему val на прогнозах train-only моделей. Test не используется. При переходе к train + val коэффициенты замораживаются; смена обучающего периода может изменить смещение модели, что и проверяется на test.


In [ ]:
def calibration_keys(frame):
    mapping = route_groups.set_index('route_id').volume_group
    group = frame.route_id.astype('int64').map(mapping)
    if group.isna().any():
        raise ValueError('Маршрут без группы объёма')
    return frame.horizon.astype(str) + ':' + group.astype(int).astype(str)

def fit_calibrator(frame, method):
    actual = frame.y.to_numpy(dtype=np.float64)
    pred = frame.prediction.to_numpy(dtype=np.float64)
    if len(frame) == 0 or pred.sum() <= 0:
        raise ValueError('Недостаточно прогнозов для калибровки')
    global_factor = float(actual.sum() / pred.sum())
    result = {'method': method, 'global_factor': global_factor}
    if method == 'grouped':
        groups = pd.DataFrame({'key': calibration_keys(frame).to_numpy(), 'y': actual, 'pred': pred}).groupby('key').agg(y=('y', 'sum'), pred=('pred', 'sum'), n=('y', 'size'))
        local = groups.y / groups.pred.replace(0, np.nan)
        weight = groups.n / (groups.n + GROUP_SHRINKAGE_ROWS)
        factors = (weight * local + (1 - weight) * global_factor).fillna(global_factor)
        result['factors'] = {str(k): float(v) for k, v in factors.items()}
    elif method == 'nonlinear':
        edges = np.unique(np.quantile(pred, np.linspace(0, 1, ISOTONIC_BINS + 1)))
        bins = np.searchsorted(edges[1:-1], pred, side='right')
        binned = pd.DataFrame({'bin': bins, 'x': pred / TARGET_SCALE, 'y': actual / TARGET_SCALE}).groupby('bin').agg(x=('x', 'mean'), y=('y', 'mean'), weight=('y', 'size')).sort_values('x')
        if len(binned) < 2:
            result['fallback_global'] = True
        else:
            iso = IsotonicRegression(y_min=0, out_of_bounds='clip')
            iso.fit(binned.x, binned.y, sample_weight=binned.weight)
            result['x'] = (iso.X_thresholds_ * TARGET_SCALE).tolist()
            result['y'] = (iso.y_thresholds_ * TARGET_SCALE).tolist()
    return result

def apply_calibrator(frame, calibrator):
    pred = frame.prediction.to_numpy(dtype=np.float64)
    method = calibrator['method']
    if method == 'none':
        result = pred.copy()
    elif method == 'global' or calibrator.get('fallback_global', False):
        result = pred * calibrator['global_factor']
    elif method == 'grouped':
        factor = calibration_keys(frame).map(calibrator['factors']).fillna(calibrator['global_factor']).to_numpy(dtype=np.float64)
        result = pred * factor
    elif method == 'nonlinear':
        x = np.asarray(calibrator['x'])
        y = np.asarray(calibrator['y'])
        result = np.interp(pred, x, y)
        lower = pred < x[0]
        upper = pred > x[-1]
        result[lower] = pred[lower] * (y[0] / x[0] if x[0] > 0 else calibrator['global_factor'])
        result[upper] = pred[upper] * (y[-1] / x[-1] if x[-1] > 0 else calibrator['global_factor'])
    else:
        raise ValueError(method)
    if not np.isfinite(result).all():
        raise ValueError('Неконечный результат калибровки')
    return np.maximum(result, 0)
selection_rows = []
selected_calibrations = {}
calibrators = {}
for variant in VARIANTS:
    panel = calibration_predictions[variant]
    fit_part = panel.loc[panel.cutoff < calibration_select_start]
    select_part = panel.loc[panel.cutoff >= calibration_select_start]
    rows = []
    for method in CALIBRATIONS:
        fitted = fit_calibrator(fit_part, method)
        pred = apply_calibrator(select_part, fitted)
        rows.append({'variant': variant, 'calibration': method, **metric_row(select_part, pred, 'calibration_select')})
    selected_method = min(rows, key=lambda row: row['score'])['calibration']
    selected_calibrations[variant] = selected_method
    calibrators[variant] = fit_calibrator(panel, selected_method)
    selection_rows.extend(rows)
selection = pd.DataFrame(selection_rows)
selection['selected_on_val'] = selection.apply(lambda row: selected_calibrations[row.variant] == row.calibration, axis=1)
selection.to_csv(RUN_DIR / 'calibration_selection.csv', index=False)
display(selection)
save_json(RUN_DIR / 'frozen_calibrators.json', {'selected': selected_calibrations, 'calibrators': calibrators, 'fit_on': 'train_only_oos_val_predictions_and_raw_val_targets', 'selection_start': str(calibration_select_start), 'test_used': False})
print('Зафиксировано до train + val:', selected_calibrations, flush=True)


## Этап 2: заново обучить на train + val и оценить test

Во всех fit метка timestamp раньше test_start. Размеченный test остаётся вне обучения. Статистический лаг переоценивается до test_start отдельно для raw и clean; train-квинтили калибровки остаются зафиксированными. На test используются уже доступные к cutoff наблюдения, включая историю прошлых test-дней; будущие относительно cutoff статусы и цели не входят в признаки.

После каждого fit показывается train + val и test по исходному таргету до и после выбранной на val калибровки. Проверка выполняется на одном наборе пар cutoff/route/horizon. Сравнение с общей фиксированной калибровкой none отделяет влияние очистки от изменения выбранного калибратора. Полного refit с включением test нет.


In [ ]:
test_predictions = {}
trainval_metric_rows = []
for variant in VARIANTS:
    signal = phase_signals[variant]
    hub_signal = signal['signal']
    refit_lag_table = calculate_dynamic_shoulder(signal['target'], hub_signal, times, routes, fit_before=test_start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
    refit_lag_table.to_csv(RUN_DIR / f'{variant}_status6_lags_trainval.csv', index=False)
    prediction_parts = []
    for h in range(1, 9):
        data = add_correlation_features(pd.read_parquet(cache_dirs[variant] / f'history_h{h}.parquet'), h, refit_lag_table)
        data['route_id'] = data.route_id.astype(route_dtype)
        columns = feature_columns[h]
        trainval = data.timestamp.lt(test_start)
        trainval1030 = trainval & data.cutoff.isin(dates[dates < test_start])
        evaluate = data.cutoff.isin(test_dates)
        if not data.loc[trainval, 'timestamp'].lt(test_start).all() or not data.loc[evaluate, 'cutoff'].ge(test_start).all():
            raise ValueError('Нарушены границы test')
        print('TRAIN+VAL', variant, f'h{h}', 'rows', int(trainval.sum()), 'max_label', str(data.loc[trainval, 'timestamp'].max()), 'calibration', selected_calibrations[variant], flush=True)
        clock = time.monotonic()
        model, train_gpu = fit_gpu_model(h, data, trainval, columns)
        model.save_model(RUN_DIR / f'{variant}_h{h}_trainval.ubj')
        train_pred = predict_gpu_matrix(model, train_gpu)
        del train_gpu
        release_gpu_pool()
        rows = [{'calibration': 'none', **metric_row(data.loc[trainval], train_pred, 'trainval_all_halfhours')}]
        train_panel = data.loc[trainval1030, ['cutoff', 'timestamp', 'route_id', 'y']].copy()
        train_panel['prediction'] = train_pred[data.loc[trainval].cutoff.isin(dates[dates < test_start]).to_numpy()]
        train_panel['horizon'] = h
        test_panel = data.loc[evaluate, ['cutoff', 'timestamp', 'route_id', 'y']].copy()
        test_panel['prediction'] = predict_gpu_frame(model, data.loc[evaluate, columns], columns)
        test_panel['horizon'] = h
        test_panel['prediction_calibrated'] = apply_calibrator(test_panel, calibrators[variant])
        for split, panel in [('trainval_1030', train_panel), ('test', test_panel)]:
            rows.append({'calibration': 'none', **metric_row(panel, panel.prediction, split)})
            rows.append({'calibration': 'selected_on_val:' + selected_calibrations[variant], **metric_row(panel, apply_calibrator(panel, calibrators[variant]), split)})
        table = pd.DataFrame(rows)
        table.insert(0, 'horizon', h)
        table.insert(0, 'variant', variant)
        trainval_metric_rows.append(table)
        display(table)
        train_panel['prediction_calibrated'] = apply_calibrator(train_panel, calibrators[variant])
        train_panel.to_parquet(RUN_DIR / f'{variant}_h{h}_trainval_1030_predictions.parquet', index=False)
        test_panel.to_parquet(RUN_DIR / f'{variant}_h{h}_test_predictions.parquet', index=False)
        prediction_parts.append(test_panel)
        print('DONE TRAIN+VAL', variant, f'h{h}', 'seconds', round(time.monotonic() - clock, 1), flush=True)
        del data, model, train_pred, train_panel
        release_gpu_pool()
    test_predictions[variant] = pd.concat(prediction_parts, ignore_index=True)
    test_predictions[variant].to_parquet(RUN_DIR / f'{variant}_test_predictions.parquet', index=False)
trainval_metrics = pd.concat(trainval_metric_rows, ignore_index=True)
trainval_metrics.to_csv(RUN_DIR / 'trainval_metrics_by_horizon.csv', index=False)


In [ ]:
comparison_rows = []
daily_rows = []
for variant in VARIANTS:
    train_panel = pd.concat([pd.read_parquet(RUN_DIR / f'{variant}_h{h}_trainval_1030_predictions.parquet') for h in range(1, 9)], ignore_index=True)
    for split, panel in [('trainval_1030', train_panel), ('test', test_predictions[variant])]:
        for kind, column in [('none', 'prediction'), ('selected_on_val:' + selected_calibrations[variant], 'prediction_calibrated')]:
            comparison_rows.append({'variant': variant, 'calibration': kind, **metric_row(panel, panel[column], split)})
    panel = test_predictions[variant]
    for cutoff, block in panel.groupby('cutoff'):
        daily_rows.append({'variant': variant, 'cutoff': str(cutoff), **metric_row(block, block.prediction_calibrated, 'test_selected_calibration')})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RUN_DIR / 'comparison_trainval_test.csv', index=False)
display(comparison)
daily = pd.DataFrame(daily_rows)
daily.to_csv(RUN_DIR / 'test_metrics_by_day.csv', index=False)
display(daily)
keys = ['cutoff', 'timestamp', 'route_id', 'horizon']
paired = test_predictions['raw'].merge(test_predictions['clean'], on=keys, how='outer', suffixes=('_raw', '_clean'), indicator=True, validate='one_to_one')
if not paired['_merge'].eq('both').all() or not np.array_equal(paired.y_raw.to_numpy(), paired.y_clean.to_numpy()):
    raise ValueError('Raw и clean оценены на разных исходных ответах')
paired['absolute_error_raw'] = np.abs(paired.prediction_calibrated_raw - paired.y_raw)
paired['absolute_error_clean'] = np.abs(paired.prediction_calibrated_clean - paired.y_raw)
paired.to_parquet(RUN_DIR / 'paired_test_predictions.parquet', index=False)
selected_test = comparison.loc[comparison.split.eq('test') & comparison.calibration.str.startswith('selected_on_val:')].set_index('variant')
delta = float(selected_test.loc['clean', 'score'] - selected_test.loc['raw', 'score'])
print('Test clean − raw, выбранные калибровки:', delta, '(меньше нуля — лучше clean)')
none_test = comparison.loc[comparison.split.eq('test') & comparison.calibration.eq('none')].set_index('variant')
print('Test clean − raw, без калибровки:', float(none_test.loc['clean', 'score'] - none_test.loc['raw', 'score']))
save_json(RUN_DIR / 'final_protocol.json', {'fit': 'train_plus_val_only', 'fit_label_before': str(test_start), 'calibration_frozen_before_trainval': True, 'selected_calibrations': selected_calibrations, 'test_used_for_fit_or_selection': False, 'full_history_refit': False, 'submissions_created': False, 'clean_minus_raw_test_score': delta})
print('Независимая от выбора на val оценка test получена после обучения на train + val, без включения test в fit. Полное переобучение на всей истории не выполнялось.')
print('Эти даты уже изучались в проекте. Отчёты и построчные прогнозы:', RUN_DIR)
